# Flux estimates vs Monte Carlo CIs

In [4]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import openpyxl
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import os

pt_orig = pd.read_excel("estimated_net_fluxes_original.xlsx", index_col=0).iloc[:, 0].rename("orig")
pt_imp = pd.read_excel("estimated_net_fluxes_imputed.xlsx", index_col=0).iloc[:, 0].rename("imp")
ci_orig = pd.read_excel("netflux_mc_CIs_original.xlsx", index_col=0); ci_orig.columns = ["LB0", "UB0"]
ci_imp = pd.read_excel("netflux_mc_CIs_imputed.xlsx", index_col=0); ci_imp.columns = ["LB1", "UB1"]
 
df = pd.concat([pt_orig, pt_imp, ci_orig, ci_imp], axis=1)

DILUTION_SUFFIX = r"_d[123]$"
df = df[~df.index.str.contains(DILUTION_SUFFIX)]
df["mag"] = np.maximum(df.orig.abs(), df.imp.abs())
df = df.sort_values("mag", ascending=False).reset_index()

chunks = np.array_split(df.set_index("index"), 6)
 
C_CI_O = "#9ec3e6"; C_PT_O = "#16436e"
C_CI_I = "#f6c98b"; C_PT_I = "#d9760a"  
OFFSET = 0.2

legend_handles = [
    Patch(fc=C_CI_O, ec=C_PT_O, label="MC CI — original"),
    Line2D([], [], marker="x", ls="", mec=C_PT_O, mew=2, ms=8,
           label="Point estimate — original"),
    Patch(fc=C_CI_I, ec=C_PT_I, label="MC CI — imputed"),
    Line2D([], [], marker="x", ls="", mec=C_PT_I, mew=2, ms=8,
           label="Point estimate — imputed"),]

/home/duford/machlearn/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [5]:
for panel_idx, chunk in enumerate(chunks, start=1):
    fig, ax = plt.subplots(figsize=(14, 4.2))
    x = np.arange(len(chunk))
 
    # CI bars
    ax.bar(x - OFFSET, chunk.UB0 - chunk.LB0, bottom=chunk.LB0, width=0.4, color=C_CI_O, edgecolor=C_PT_O, lw=0.4, zorder=2)
    ax.bar(x + OFFSET, chunk.UB1 - chunk.LB1, bottom=chunk.LB1, width=0.4, color=C_CI_I, edgecolor=C_PT_I, lw=0.4, zorder=2)
 
    # CI cap lines
    for xi, lb, ub in zip(x - OFFSET, chunk.LB0, chunk.UB0):
        ax.hlines([lb, ub], xi - 0.2, xi + 0.2, color=C_PT_O, lw=0.6, zorder=3)
    for xi, lb, ub in zip(x + OFFSET, chunk.LB1, chunk.UB1):
        ax.hlines([lb, ub], xi - 0.2, xi + 0.2, color=C_PT_I, lw=0.6, zorder=3)
 
    # Point estimates
    ax.scatter(x - OFFSET, chunk.orig, marker="x", s=60, c=C_PT_O, linewidths=2.1, zorder=6)
    ax.scatter(x + OFFSET, chunk.imp,   marker="x", s=60, c=C_PT_I, linewidths=2.1, zorder=6)

    ax.axhline(0, color="#888", lw=0.7, zorder=1)
    ax.set_xticks(x)
    ax.set_xticklabels(chunk.index, rotation=45, ha="right", fontsize=11)
    ax.set_ylabel("Net flux (µmol/gDW)", fontsize=11)
    ax.tick_params(axis="y", labelsize=10)
    ax.set_facecolor("#eaeaf2")
    ax.grid(axis="y", color="white", lw=0.8, zorder=0)
    ax.set_xlim(-0.7, len(chunk) - 0.3)
    ax.legend(handles=legend_handles, ncol=2, fontsize=9, loc="upper right", framealpha=0.95)
 
    fig.tight_layout()
    out_path = os.path.join(f"flux_CI_panel_{panel_idx}.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

# Flux ranges vs point estimate locations with respect to CIs in the imputed set

In [21]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pt = pd.read_excel("estimated_net_fluxes_imputed.xlsx", index_col=0).iloc[:, 0]
ci = pd.read_excel("netflux_mc_CIs_imputed_deep.xlsx",  index_col=0).set_axis(["lb","ub"], axis=1)
raw = pd.read_csv("fluxes_fva.csv", index_col=0)
raw = raw.drop(['Var2', 'Var3'], axis=1)

fwd = [r for r in raw.index if r.endswith("_f")]
bwd_set = set(r for r in raw.index if r.endswith("_b"))
irr = [r for r in raw.index if not r.endswith(("_f","_b"))]

net = {}
for r in irr:
    net[r] = raw.loc[r].values
for r in fwd:
    base = r[:-2]
    net[base] = raw.loc[r].values - (raw.loc[base+"_b"].values if base+"_b" in bwd_set else 0)
    
fc = pd.DataFrame(net).T

ad = ci.join(pt.rename("est")).dropna()
ad = ad[~ad.index.str.contains(r"_d[123]$")]
ad["inside"] = ad.est.between(ad.lb, ad.ub)
ad["offset"] = (ad.est - (ad.lb+ad.ub)/2).abs() / (ad.ub - ad.lb)
fc_vals = fc.reindex(ad.index).values
ad["flux_min"] = fc_vals.min(axis=1)
ad["flux_max"] = fc_vals.max(axis=1)
print(ad)
ad["flux_range"] = ad.flux_max - ad.flux_min
ad["crosses_zero"] = (ad.flux_min < 0) & (ad.flux_max > 0)

top4 = ad[~ad.inside].nlargest(4, "offset").index.tolist()
print("Top 4 outliers:", top4)

fig, ax = plt.subplots(figsize=(7, 5))
for inside, color, label in [(False, "#d6604d", "Outside CI"), (True, "#2166ac", "Inside CI")]:
    g = ad[ad.inside == inside]
    ax.scatter(g.flux_range, g.offset, c=color, s=55, alpha=0.8, edgecolors="white", lw=0.4, label=label, zorder=3)

zc = ad[ad.crosses_zero]
ax.scatter(zc.flux_range, zc.offset, marker="+", s=80, c="black", linewidths=1.2, zorder=5, label="Flux crosses zero")

ax.axhline(0.5, color="#555", lw=0.8, ls="--", label="CI boundary (0.5)")

rho, p = stats.spearmanr(ad.flux_range, ad.offset.fillna(0))
ax.text(0.97, 0.97, f"Spearman ρ = {rho:+.3f}\np = {p:.1e}", transform=ax.transAxes, ha="right", va="top", fontsize=9, bbox=dict(fc="white", ec="#ccc", pad=3))
nudge = {
    "T_PEP": ( 0.03,  0.10),
    "T_T3P": (-0.01,  0.18), 
    "FGAM_c":( 0.03, -0.18),  
    "ENO_h": ( 0.03, -0.15),
}
for rxn in top4:
    dx, dy = nudge.get(rxn, (0.02, 0.10))
    ax.annotate(rxn, xy=(ad.loc[rxn,"flux_range"], ad.loc[rxn,"offset"]), xytext=(ad.loc[rxn,"flux_range"]+dx, ad.loc[rxn,"offset"]+dy),
                fontsize=9, arrowprops=dict(arrowstyle="->", color="#555", lw=0.9))

ax.set(xlabel="Flux range (µmol/gDW)", ylabel="|Offset from CI centre| / CI width")
ax.legend(fontsize=9)
ax.set_facecolor("#f5f5f5")
fig.tight_layout()
fig.savefig("ci_containment_constrained.png", dpi=150, bbox_inches="tight")
print(f"Spearman rho={rho:.3f}, p={p:.2e}")

                     lb        ub       est  inside    offset  flux_min  \
RBPCh          0.222356  0.433800  0.505149   False  0.837435  0.078976   
GAPDH_nadp_hi  0.377074  0.797490  0.968050   False  0.905692  0.135205   
ALDh           0.003464  0.273969  0.383807   False  0.906048  0.000022   
SBPase         0.003464  0.273969  0.383807   False  0.906048  0.000022   
FBAh           0.004691  0.289747  0.015997    True  0.460336  0.000014   
...                 ...       ...       ...     ...       ...       ...   
GLUtm         -0.178558 -0.022317 -0.010007   False  0.578788 -0.850961   
ALAtm         -0.168573  0.039350  0.009663    True  0.357222 -0.726447   
PEPtm          0.001054  0.038839  0.000192   False  0.522793  0.000002   
CO2tm          0.010910  0.042287  0.010408   False  0.515991  0.000381   
AKGtm         -0.178038 -0.022195 -0.008410   False  0.588457 -0.850433   

               flux_max  
RBPCh          0.599484  
GAPDH_nadp_hi  0.999958  
ALDh           0.4006

# Flux ranges

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

df = pd.read_csv('fluxes_fva.csv')
scols = [c for c in df.columns if c.startswith('Var4_')]

low_df  = df[df['Var3'] <  0.01].reset_index(drop=True)
high_df = df[df['Var3'] >= 0.01].reset_index(drop=True)

def pages(src): return [src['Var1'].tolist()[i:i+PER_PAGE] for i in range(0, len(src), 10)]

hp, lp = pages(high_df), pages(low_df)
total  = len(hp) + len(lp)

def plot(rxns, src, page_num):
    d = src.set_index('Var1').reindex(rxns)
    samples = [d.loc[r, scols].values.astype(float) for r in rxns]
    
    max_val = d['Var3'].max()
    mag = int(np.floor(np.log10(max_val))) if max_val > 0 else 0
    scale = 10 ** (-mag) if mag < -1 else 1
    ylabel = f'Flux (\u00d710\u207b{-mag})' if mag < -1 else 'Flux'

    fig, ax = plt.subplots(figsize=(14, 6))
    pos = np.arange(1, len(rxns) + 1)
    ax.boxplot([s * scale for s in samples], positions=pos, widths=0.5, patch_artist=True,
               flierprops=dict(marker='o', markersize=3, markerfacecolor='lightgray',
                               markeredgecolor='gray', linestyle='none', alpha=0.6),
               medianprops=dict(color='black', linewidth=1.5),
               boxprops=dict(facecolor='white'), whiskerprops=dict(color='black'),
               capprops=dict(color='black'))
    for p, fmin, fmax in zip(pos, d['Var2'] * scale, d['Var3'] * scale):
        ax.hlines(fmin, p - 0.4, p + 0.4, colors='green', linewidths=2, zorder=5)
        ax.hlines(fmax, p - 0.4, p + 0.4, colors='red',   linewidths=2, zorder=5)
    ax.set_xticks(pos)
    ax.set_xticklabels(rxns, rotation=30, ha='right', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xlim(0.5, len(rxns) + 0.5)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.ticklabel_format(style='plain', axis='y')
    ax.legend(handles=[mpatches.Patch(color='green', label='FVA min'), mpatches.Patch(color='red',   label='FVA max')], loc='upper right', fontsize=13)
    plt.tight_layout()
    path = os.path.join(f'flux_range_{page_num:02d}_of_{total}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')

for i, rxns in enumerate(hp): 
    plot(rxns, high_df, i + 1)
for i, rxns in enumerate(lp): 
    plot(rxns, low_df,  len(hp) + i + 1)

Saved flux_range_01_of_11.png
Saved flux_range_02_of_11.png
Saved flux_range_03_of_11.png
Saved flux_range_04_of_11.png
Saved flux_range_05_of_11.png
Saved flux_range_06_of_11.png
Saved flux_range_07_of_11.png
Saved flux_range_08_of_11.png
Saved flux_range_09_of_11.png
Saved flux_range_10_of_11.png
Saved flux_range_11_of_11.png


In [20]:
k = df.drop(['Var2', 'Var3'], axis=1)
k

,Var1,Var4_1,Var4_2,Var4_3,Var4_4,Var4_5,Var4_6,Var4_7,Var4_8,Var4_9,...,Var4_7991,Var4_7992,Var4_7993,Var4_7994,Var4_7995,Var4_7996,Var4_7997,Var4_7998,Var4_7999,Var4_8000
0,RBPCh,0.439501,0.433485,0.546193,0.361756,0.417159,0.576737,0.507722,0.432291,0.398127,...,0.582190,0.506443,0.526336,0.555420,0.358975,0.425340,0.403834,0.475063,0.499865,0.465694
1,GAPDH_nadp_hi,0.775274,0.744611,0.919679,0.615825,0.702701,0.997727,0.881980,0.739707,0.675899,...,0.987860,0.845277,0.885497,0.936283,0.616145,0.709994,0.675316,0.796391,0.840872,0.796050
2,ALDh_f,0.928089,0.935112,0.894671,0.755161,0.885767,0.347756,0.422837,0.642722,0.807256,...,0.884047,0.738793,0.507836,0.244973,0.274980,0.951785,0.955837,0.789680,0.582079,0.395540
3,SBPase,0.080681,0.094024,0.124150,0.050187,0.111440,0.203405,0.120439,0.031814,0.017961,...,0.240165,0.160447,0.121297,0.173101,0.087516,0.165540,0.160551,0.161338,0.143074,0.239921
4,FBAh_f,0.752264,0.805494,0.603819,0.334763,0.749385,0.856976,0.628756,0.735581,0.879277,...,0.945900,0.728906,0.850507,0.761759,0.837135,0.535361,0.398009,0.685272,0.818544,0.619323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,GLUtm_b,0.677967,0.518556,0.914836,0.848631,0.736787,0.307927,0.301364,0.341488,0.568733,...,0.813127,0.961257,0.517695,0.947033,0.566710,0.440628,0.311379,0.455368,0.811793,0.678141
101,ALAtm_b,0.733829,0.478675,0.325276,0.410311,0.649782,0.688826,0.601427,0.656679,0.718405,...,0.195756,0.374739,0.883498,0.992273,0.891605,0.638081,0.461924,0.479225,0.869333,0.773818
102,PEPtm_b,0.187419,0.043518,0.064234,0.517436,0.843562,0.709280,0.791129,0.885133,0.486205,...,0.326596,0.208063,0.106181,0.027842,0.367771,0.167356,0.318174,0.188030,0.373321,0.152447
103,CO2tm_b,0.991437,0.840495,0.893393,0.822101,0.358562,0.266010,0.520817,0.393611,0.833809,...,0.116431,0.677614,0.096093,0.055269,0.018456,0.138583,0.190453,0.223127,0.520232,0.025987
